# Getting Started with `qox` 

`qox` is a quantitative finance library designed for high-performance option pricing and risk management. This notebook demonstrates the core workflow: defining an instrument, setting up market data, and computing prices and Greeks.

In [ ]:
!pip install qox

## 1. Setup and Inputs

We begin by defining our environment. `qox` utilizes Python's native `zoneinfo` to ensure time-aware calculations.

In [ ]:
import time
from datetime import datetime
from zoneinfo import ZoneInfo

import qox

ny_tz = ZoneInfo("America/New_York")
spot = 95.0
strike = 100.0
valuation_time = datetime(2025, 9, 25, 17, 0, tzinfo=ny_tz)
expiry = datetime(2026, 9, 25, 17, 0, tzinfo=ny_tz)
rate = 0.05
vol = 0.2
option_type = qox.OptionType.Call
exercise_style = qox.ExerciseStyle.European

stock_option = qox.StockOption(strike, expiry, option_type, exercise_style)

## 2. Defining the Market Frame

The `OptionMarketFrame` encapsulates the state of the market. While future versions will support complex rate curves and volatility surfaces, we currently utilize continuous rates and flat volatility.

In [ ]:
market_frame = qox.OptionMarketFrame(
    spot=spot,
    rate_curve=qox.RateCurve.continuous(rate, qox.DayCountConvention.Act365Fixed),
    vol_surface=qox.VolSurface.flat(vol, qox.DayCountConvention.Act365Fixed),
)

## 3. Basic Valuation (Default Analytic)

When no configuration is specified, `qox` reverts to its internal defaults. For European options, this means using the Black-Scholes analytic solution.

The `.at()` method is used here to set the valuation time; if omitted, it defaults to the current system time.

In [ ]:
result = (
    stock_option.valuation()
    .at(valuation_time)
    .market(market_frame)
    .compute()
)

print(f"Price: {result.price:.4f}")
print("-" * 20)
print("Greeks:")
print(f"  Delta: {result.greeks.delta:.5f}")
print(f"  Gamma: {result.greeks.gamma:.5f}")
print(f"  Theta: {result.greeks.theta:.5f}")
print(f"  Vega:  {result.greeks.vega:.5f}")
print(f"  Rho:   {result.greeks.rho:.5f}")

## 4. Advanced: Finite Difference Method (FDM)

You can override the default analytical engine by providing a specific `Config`. Note that Vega and Rho are not yet supported for FDM.

In [ ]:
fdm_config = qox.FdmConfig(nodes=1000, time_steps=50, grid_std_devs=6.0)
config = qox.Config().add_policy(qox.InstrumentPolicy().european().fdm(fdm_config))

fdm_result = (
    stock_option.valuation()
    .at(valuation_time)
    .market(market_frame)
    .config(config)
    .compute()
)

print(f"Price: {fdm_result.price:.4f}")
print("-" * 20)
print("Greeks:")
print(f"  Delta: {fdm_result.greeks.delta:.5f}")
print(f"  Gamma: {fdm_result.greeks.gamma:.5f}")
print(f"  Theta: {fdm_result.greeks.theta:.5f}")